In [ ]:
import pandas as pd
import ast
import unicodedata
import string

In [ ]:
filename = '/Users/graven2/Documents/THESIS/YorubaThesis/Combined/ToneThenDotsFinalEval.csv'
df = pd.read_csv(filename, header=0, index_col=0, converters={'Prediction':ast.literal_eval})
def func(row):
    try: return row['Wrong Words']/row['Total Words']
    except: return 0
df['WER'] = df.apply(lambda row: func(row), axis=1)
df

# Get Rows for Analysis

In [ ]:
# Get 10 worst performing rows
high_wer = df.nlargest(10, 'WER')
high_wer.to_csv('FINAL_high_wer.csv')

# Get 10 random rows
random = df.sample(10)
random.to_csv('FINAL_random10.csv')

In [ ]:
high_wer

# Do same analysis as LLMs

In [ ]:
def syll2text(syllables):
    combined_sylls = []
    # look for SP and UNK syllables
    for syll in syllables:
        if syll[0] in ['SP', 'UNK']: syll = syll[1:]
        combined_sylls.append(''.join(syll))
    final = ''.join(combined_sylls)
    if not final: return ' '
    else: return final


df['Predicted Sentence'] = df.apply(lambda row: syll2text(row['Prediction']), axis=1)


In [ ]:
LOTONE = chr(0x0300)
HITONE = chr(0x0301)
RISETONE = chr(0x030C)
MIDTONE1 = chr(0x0304)
MIDTONE2 = chr(0x0305)
TONECHARS = {LOTONE, HITONE, RISETONE,MIDTONE1,MIDTONE2}

UNDERDOT = chr(0x0323)
UNDERLINE = chr(0x0329)
UNDERDIACS = {UNDERDOT, UNDERLINE}

def remove_diacritics(input_str):
    '''
    Removes diacritical marks from a Unicode string.
    '''
    # normalize and fully decompose
    nfkd_form = unicodedata.normalize('NFD', input_str.lower())

    # select all non-combining characters
    non_combining = [c for c in nfkd_form if not unicodedata.combining(c)]
    return ''.join(non_combining)

def normalize(input_str):
    '''
    Normalize into NFD form (fully decomposed)
    Also convert everything to lowercase
    '''
    nfd = unicodedata.normalize('NFD', input_str.lower())


    # convert underline to underdot
    return nfd.replace(chr(0x0329), chr(0x0323))

def eval_row(row):
    '''
    Counts errors in a row

    Output: number of hallucinated words, 
    number of wrong words, 
    total number of words
    '''
    pred = row['Predicted Sentence']
    target = str(row['Sentence'])
    # source = row['Sources']
    pred_words = pred.split()
    target_words = target.split()
    # source_words = source.split(' ')

    # if number of words doesn't match, return NaN
    if len(pred_words) != len(target_words): 
        curr_id = row['ID']
        print(f'\n{curr_id}')
        print(pred, target)
        print(pred_words, target_words)
        default = -1
        return pd.Series({'Hallucinations' : default,  'L as L': default, 'L as M' : default, \
                        'L as H' : default, 'M as L' : default, 'M as M':default, 'M as H': default, 'H as L': default, 'H as M': default, 'H as H': default, \
                            'Missing Dot': default, 'Extra Dot': default, 'Dot as Dot': default, 'Dotless as Dotless' : default,\
                                'Other Error': default})

    # calculate hallucinations and wrong diacritics
    total = 0
    hallucinations = 0
    wrong = 0
    LasL = 0
    LasM = 0
    LasH = 0
    MasL = 0
    MasM = 0
    MasH = 0
    HasL = 0
    HasM = 0
    HasH = 0
    missingDot = 0
    extraDot = 0
    otherMistake = 0
    dotAsDot = 0
    dotlessAsDotless = 0
    for i in range(len(pred_words)):
        curr_pred_word = pred_words[i]
        curr_target_word = target_words[i]
        # curr_source_word = source_words[i]
        
        # if word is punctuation, don't count it
        if len(curr_target_word) == 1 and curr_target_word in string.punctuation: continue
        else: total+=1

        # look for hallucination
        no_diacs_pred = remove_diacritics(curr_pred_word)
        no_diacs_true = remove_diacritics(curr_target_word)
        if no_diacs_pred != no_diacs_true:
            hallucinations+=1
            wrong+=1
            continue

        # look for wrong diacritics
        normal_pred = normalize(curr_pred_word)
        normal_true = normalize(curr_target_word)
        if normal_pred == normal_true:
            for i in range(len(normal_pred)):
                char = normal_pred[i]
                if char == HITONE: HasH+=1
                elif char == LOTONE: LasL +=1
                elif char == UNDERDOT: dotAsDot +=1
                elif char in ['s', 'e', 'o']:
                    if i < len(normal_pred) - 1:
                        if normal_pred[i+1] != UNDERDOT: dotlessAsDotless+=1
                    else:  dotlessAsDotless+=1
                if char in ['a', 'e', 'i', 'o', 'u']:
                    if i < len(normal_pred) - 2:
                        if normal_pred[i+1] not in TONECHARS and normal_pred[i+2] not in TONECHARS: MasM+=1
                    elif i < len(normal_pred) - 1:
                        if normal_pred[i+1] not in TONECHARS: MasM+=1
                    else:  MasM+=1
        if normal_pred != normal_true:
            wrong+=1
            # loop through word and find what is different
            i = 0 # pred index
            j = 0 # true index
            while i < len(normal_pred) and j < len(normal_true):
                pred_char = normal_pred[i]
                true_char = normal_true[j]
                # same char
                if pred_char == true_char:
                    if pred_char == HITONE: HasH+=1
                    elif pred_char == LOTONE: LasL +=1
                    elif pred_char == UNDERDOT: dotAsDot +=1
                    elif pred_char in ['s', 'e', 'o']:
                        if i < len(normal_pred) - 1:
                            if normal_pred[i+1] != UNDERDOT: dotlessAsDotless+=1
                        else:  dotlessAsDotless+=1
                    if pred_char in ['a', 'e', 'i', 'o', 'u']:
                        if i < len(normal_pred) - 2:
                            if normal_pred[i+1] not in TONECHARS and normal_pred[i+2] not in TONECHARS: MasM+=1
                        elif i < len(normal_pred) - 1:
                            if normal_pred[i+1] not in TONECHARS: MasM+=1
                        else:  MasM+=1
                    i+=1
                    j+=1
                # pred is a letter, true is a diacritic
                elif (unicodedata.combining(pred_char) == 0) and (unicodedata.combining(true_char) != 0):
                    if true_char == HITONE: HasM += 1
                    if true_char == LOTONE: LasM += 1
                    if true_char == UNDERDOT: missingDot += 1
                    j+=1
                # pred is diacritic, true is a letter
                elif (unicodedata.combining(pred_char) != 0) and (unicodedata.combining(true_char) == 0):
                    if pred_char == HITONE: MasH +=1
                    if pred_char == LOTONE: MasL +=1
                    if pred_char == UNDERDOT: extraDot += 1
                    i+=1
                # both are diacritics (more complex)
                # check for dots first
                elif pred_char == UNDERDOT:
                    extraDot+=1
                    i+=1
                elif true_char == UNDERDOT:
                    missingDot+=1
                    j+=1
                # now compare tones
                elif pred_char == HITONE and true_char == LOTONE:
                    LasH +=1
                    i+=1
                    j+=1
                elif pred_char == LOTONE and true_char == HITONE:
                    HasL += 1
                    i+=1
                    j+=1
                else:
                    print(pred_char, true_char)
                    print(curr_pred_word, curr_target_word)
                    otherMistake+=1
                    i+=1
                    j+=1
            if i < len(curr_pred_word):
                if curr_pred_word[i] == HITONE: MasH+=1
                elif curr_pred_word[i] == LOTONE: MasL+=1
                elif curr_pred_word[i] == UNDERDOT: extraDot +=1
                else: otherMistake+=1
            if j < len(curr_target_word):
                if curr_target_word[j] == HITONE: HasM+=1
                elif curr_target_word[j] == LOTONE: LasM +=1
                elif curr_target_word[j] == UNDERDOT: missingDot+=1
                else: otherMistake+=1

    return pd.Series({'Hallucinations' : hallucinations, 'L as L': LasL, 'L as M' : LasM, \
                        'L as H' : LasH, 'M as L' : MasL, 'M as M':MasM, 'M as H': MasH, 'H as L': HasL, 'H as M': HasM, 'H as H': HasH, \
                            'Missing Dot': missingDot, 'Extra Dot': extraDot, 'Dot as Dot': dotAsDot, 'Dotless as Dotless' : dotlessAsDotless,\
                                'Other Error': otherMistake})
   

In [ ]:
eval_row(df.iloc[1152])
print(df.iloc[1152])


In [ ]:
df = df.join(df.apply(lambda row: eval_row(row), axis=1))


In [ ]:
df

In [ ]:
# find ones where lengths differed
neg_df = df[df['Total Words'] < 0]
pos_df = df[df['Total Words'] >= 0]

print(f"Wrong number of words: {len(neg_df)}\nCorrect Number of Words: {len(pos_df)}\n")

# print(f"Hallucination WER: {pos_df['Hallucination WER'].mean()}")
print(f"Full WER: {df['WER'].mean()}")
print(f"Full WER (by words): {df['Wrong Words'].sum() / df['Total Words'].sum()}")

tone_results = pd.DataFrame([[df['L as L'].sum(), df['L as M'].sum(), df['L as H'].sum()], 
              [df['M as L'].sum(), df['M as M'].sum(), df['M as H'].sum()], 
              [df['H as L'].sum(), df['H as M'].sum(), df['H as H'].sum()]], 
              columns=['Pred L', 'Pred M', 'Pred H'], index=['True L', 'True M', 'True H'])
display(tone_results)
tone_results = 100*(tone_results.div(tone_results.sum(axis=1), axis=0))
display(tone_results)

dot_results = pd.DataFrame([[df['Dot as Dot'].sum(), df['Missing Dot'].sum()], [df['Extra Dot'].sum(), df['Dotless as Dotless'].sum()]], 
                           columns=['Pred Dot', 'Pred Dotless'], index=['True Dot', 'True Dotless'])
display(dot_results)
dot_results = 100*dot_results.div(dot_results.sum(axis=1), axis=0)
display(dot_results)

# Graphing

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

In [ ]:
df['Total Orig Words'] = df.apply(lambda x: len(x['Sentence'].split()), axis=1)
# display(df)
print(df['Total Orig Words'].max())
print(len(df))
print(len(df[df['Total Orig Words'] >= 50]))

In [ ]:
test_val = 0.8

subset = df[df['WER'] >= test_val]
print(f"Length of Sentence: {subset['Total Words'].mean()}")

wrong_count = len(subset)
total_count = len(df)
print(wrong_count, total_count)
print(wrong_count/total_count)

In [ ]:
tone_results_raw = pd.DataFrame([
    [df['L as L'].sum(), df['L as M'].sum(), df['L as H'].sum()],
    [df['M as L'].sum(), df['M as M'].sum(), df['M as H'].sum()],
    [df['H as L'].sum(), df['H as M'].sum(), df['H as H'].sum()]
], columns=['Pred L', 'Pred M', 'Pred H'], index=['True L', 'True M', 'True H'])
display(tone_results_raw)

tone_results_pct = 100 * (tone_results_raw.div(tone_results_raw.sum(axis=1), axis=0))
display(tone_results_pct)
# Format percentages as annotation labels
# annot_labels = tone_results_pct.applymap(lambda x: f"{x:.1f}%")
annot_labels = pd.DataFrame(
    [[f"{int(tone_results_raw.iloc[i,j])}\n({tone_results_pct.iloc[i,j]:.2f}%)" 
      for j in range(3)] for i in range(3)],
    columns=tone_results_raw.columns,
    index=tone_results_raw.index
)

# Draw heatmap — colored by raw counts, labeled with percentages
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    tone_results_raw,          # color scale based on raw counts
    # annot=annot_labels,        # display percentages
    annot=tone_results_raw,
    annot_kws={'size': 14},
    fmt='',                    # needed when annot is strings
    cmap='Blues',
    linewidths=0.5,
    linecolor='#cccccc',
    ax=ax,
    cbar_kws={'label': 'Count'}
)

ax.set_title('Tone Confusion Matrix', fontsize=16, pad=10)
ax.set_xlabel('Predicted Diacritic', fontsize=14)
ax.set_ylabel('True Diacritic', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
dot_results_raw = pd.DataFrame([[df['Dot as Dot'].sum(), df['Missing Dot'].sum()], [df['Extra Dot'].sum(), df['Dotless as Dotless'].sum()]], 
                           columns=['Pred Dot', 'Pred Dotless'], index=['True Dot', 'True Dotless'])
display(dot_results_raw)

dot_results_pct = 100*dot_results.div(dot_results.sum(axis=1), axis=0)
display(dot_results_pct) # Format percentages for annotation labels

# annot_labels = tone_results_pct.applymap(lambda x: f"{x:.1f}%")
annot_labels = pd.DataFrame(
    [[f"{int(dot_results_raw.iloc[i,j])}\n({dot_results_pct.iloc[i,j]:.2f}%)" 
      for j in range(2)] for i in range(2)],
    columns=dot_results_raw.columns,
    index=dot_results_raw.index
)

# Draw heatmap — colored by raw counts, labeled with percentages
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    dot_results_raw,          # color scale based on raw counts
    annot=dot_results_raw,
    annot_kws={"size": 14},
    fmt='',                    # needed when annot is strings
    cmap='Blues',
    linewidths=0.5,
    linecolor='#cccccc',
    ax=ax,
    cbar_kws={'label': 'Count'},
    
)

ax.set_title('Underdots Confusion Matrix', fontsize=16, pad=10)
ax.set_xlabel('Predicted Diacritic', fontsize=14)
ax.set_ylabel('True Diacritic', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Group by 10% bins
bins = np.arange(0, 1.1, 0.1)
bin_labels = [f"{int(b*100)}-{int((b+0.1)*100)}%" for b in bins[:-1]]

# count WER
counts, _ = np.histogram(df['WER'], bins=bins)

# graph
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(bin_labels, counts)

# Add count labels on top of bars
for bar, count in zip(bars, counts):
    if count > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            str(count),
            ha='center', va='bottom'
        )

ax.set_title('WER Distribution by Sentences')
ax.set_xlabel('WER (%)')
ax.set_ylabel('Number of Sentences')
plt.show()